# Занятие 4. Работа с биометрическими датасетами

**Цель работы:** Научиться загружать, визуализировать и подготавливать открытые биометрические датасеты (лиц и отпечатков пальцев) для дальнейшего использования в лабораторных работах, а также выполнять корректное разбиение `train/test split` по субъектам.

## Шаги 1–2. Подготовка и загрузка датасетов лиц (`ORL Faces`) и отпечатков пальцев (`Fingerprints FVC`)

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath('../src'))
from load_dataset import (
    prepare_orl_faces_dataset,
    prepare_fingerprint_dataset,
    load_face_dataset,
    show_samples,
    compute_dataset_statistics,
    plot_class_distributions,
    split_by_subjects,
    plot_split_comparison
)

orl_dir = prepare_orl_faces_dataset('../datasets/orl_faces')
fp_dir = prepare_fingerprint_dataset('../datasets/fingerprints_fvc')

X_faces, y_faces = load_face_dataset(orl_dir)
X_fp, y_fp = load_face_dataset(fp_dir)

print(f'ORL Faces: всего изображений = {len(X_faces)}, субъектов = {len(set(y_faces))}')
print(f'Fingerprints FVC: всего изображений = {len(X_fp)}, субъектов = {len(set(y_fp))}')

ORL Faces: всего изображений = 400, субъектов = 40
Fingerprints FVC: всего изображений = 255, субъектов = 25


## Шаг 3. Визуализация примеров 4×4 обоих датасетов

In [2]:
show_samples(X_faces, y_faces, n=16, output_path='../reports/dataset_samples.png',
             title='Датасет лиц ORL Faces — 16 случайных образцов (4x4)')
show_samples(X_fp, y_fp, n=16, output_path='../reports/fingerprint_samples.png',
             title='Датасет отпечатков Fingerprints FVC — 16 случайных образцов (4x4)')

Визуализация 4x4 сохранена: ../reports/dataset_samples.png
Визуализация 4x4 сохранена: ../reports/fingerprint_samples.png


## Шаг 4. Статистика датасетов и гистограммы распределения по классам

In [3]:
stats_faces = compute_dataset_statistics(X_faces, y_faces, name='ORL / AT&T Faces (Лица)')
stats_fp = compute_dataset_statistics(X_fp, y_fp, name='Fingerprints FVC (Отпечатки пальцев)')
plot_class_distributions(stats_faces, stats_fp, output_path='../reports/dataset_class_distribution.png')

СТАТИСТИКА ДАТАСЕТА: ORL / AT&T Faces (Лица)
  Всего изображений:            400
  Число уникальных субъектов:   40
  Образцов на субъекта:         min=10, max=10, mean=10.00 ± 0.00
  Разрешение изображений:       92x112 (H=112, W=92)
  Средняя яркость пикселей:     139.0 ± 44.0
СТАТИСТИКА ДАТАСЕТА: Fingerprints FVC (Отпечатки пальцев)
  Всего изображений:            255
  Число уникальных субъектов:   25
  Образцов на субъекта:         min=6, max=14, mean=10.20 ± 2.71
  Разрешение изображений:       128x128 (H=128, W=128)
  Средняя яркость пикселей:     203.9 ± 63.6
Гистограмма распределения по классам сохранена: ../reports/dataset_class_distribution.png


## Шаг 5. Train/Test Split по субъектам (80/20 Subject-Disjoint Split)

In [4]:
X_tr, X_te, y_tr, y_te, tr_subj, te_subj = split_by_subjects(X_faces, y_faces, test_size=0.2, random_state=42)
X_fp_tr, X_fp_te, y_fp_tr, y_fp_te, _, _ = split_by_subjects(X_fp, y_fp, test_size=0.2, random_state=42)

plot_split_comparison(X_faces, y_faces, tr_subj, te_subj, output_path='../reports/subject_split_comparison.png')


РЕЗУЛЬТАТ РАЗБИЕНИЯ ПО СУБЪЕКТАМ (80/20 Subject-Disjoint Split):
  Всего субъектов:   40 -> Train: 32 суб., Test: 8 суб.
  Всего изображений: 400 -> X_train: 320 изобр., X_test: 80 изобр.
  Subject IDs в Test: [4, 12, 15, 16, 19, 26, 27, 37]
  Пересечение субъектов (Train ∩ Test): 0 (утечка данных отсутствует!)

РЕЗУЛЬТАТ РАЗБИЕНИЯ ПО СУБЪЕКТАМ (80/20 Subject-Disjoint Split):
  Всего субъектов:   25 -> Train: 20 суб., Test: 5 суб.
  Всего изображений: 255 -> X_train: 191 изобр., X_test: 64 изобр.
  Subject IDs в Test: [0, 8, 11, 16, 23]
  Пересечение субъектов (Train ∩ Test): 0 (утечка данных отсутствует!)
Диаграмма сравнения стратегий разбиения сохранена: ../reports/subject_split_comparison.png


## Контрольные вопросы

### 1. Почему нельзя случайно перемешивать изображения одного субъекта между train и test?
Случайное разбиение по изображениям приводит к **утечке биометрической идентичности (Subject-level Data Leakage)**: модель запоминает конкретных людей, освещение и фон из `Train` и показывает искусственно завышенную точность на других кадрах тех же людей в `Test`, теряя способность обобщаться на новых пользователей в режиме верификации 1:1.

### 2. Какие искажения (шум, освещение, ракурс) присутствуют в выбранном датасете?
- В **ORL Faces**: повороты и наклоны головы до $\pm 20^\circ$, мимика (улыбка, открытые/закрытые глаза), наличие/отсутствие очков, изменения направления освещения и масштаба.
- В **Fingerprints FVC**: вращение и сдвиг пальца на сканере, различное давление прижима (сухой/влажный отпечаток), высокочастотный шум оптического сенсора.

### 3. Чем отличается закрытый набор классов (closed-set) от открытого (open-set) применительно к тестовому разбиению?
- В **Closed-Set** множества субъектов в `Train` и `Test` совпадают ($\mathcal{C}_{\text{test}} \subseteq \mathcal{C}_{\text{train}}$), тестируется классификация среди фиксированного списка известных лиц.
- В **Open-Set** множества субъектов в `Train` и `Test` не пересекаются ($\mathcal{C}_{\text{train}} \cap \mathcal{C}_{\text{test}} = \emptyset$), тестируется способность модели извлекать универсальные признаки для сравнения ранее неизвестных системе людей (верификация 1:1).